# Python Shiny - App Widgets
In this dashboard, we take the code developed in the app_template.ipynb notebook and make it interactive through the use of **ipywdgets**. 

https://ipywidgets.readthedocs.io/en/stable/index.html

## Step 1: Read in the data and create the plot function
We use the code created in the previous notebook.

In [ ]:
# Import packages
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# Set sns theme
sns.set_theme(style='whitegrid')

In [ ]:
# Read the data into a dataframe, pivotting nutrients into longer format
df_nutrients = pd.read_csv(
    'Data/NTL-LTER_Lake_Nutrients_PeterPaul_Processed.csv',
    parse_dates=['sampledate'],
    dtype={'lakename': 'category', 'depth_id': 'category'}
).melt(
    id_vars=['lakename', 'sampledate', 'depth_id'],
    value_vars=['tn_ug', 'tp_ug', 'nh34', 'no23'],
    var_name='nutrient',
    value_name='ug'
).rename(columns={
    'lakename':'Lake name',
    'depth_id':'Depth zone'
})

#Update nutrient values to more readable names
nutrient_name_updates = {
    'tn_ug':'Total Nitrogen',
    'tp_ug':'Total Phosphorus',
    'nh34':'Ammonia',
    'no23':'Nitrates'
}
df_nutrients['nutrient'] = df_nutrients['nutrient'].map(nutrient_name_updates)

In [ ]:
### Plot as a function
def plot_nutrients(start_date = "1995-01-01",
                   end_date = "1999-12-31",
                   nutrient = "Total Nitrogen",
                   depth_zones = [1,2,3],
                   lake_names = ['Peter Lake', 'Paul Lake']):
    # Subset the data
    df_subset = df_nutrients.loc[
        (df_nutrients['sampledate'] >= start_date) &  # Start date
        (df_nutrients['sampledate'] <= end_date) &    # End date
        (df_nutrients['nutrient'] == nutrient) &  # Nutrient
        (df_nutrients['Depth zone']).isin(depth_zones) & # Depth IDs
        (df_nutrients['Lake name']).isin(lake_names), # Lake Names
    ]

    #Create a plot of selected nutrients by sampledate, with markers colored by depth_id using the inverse mako colormap and different shapes for lakename
    the_plot = sns.relplot(
        data = df_subset,
        x='sampledate',
        y='ug',
        hue='Depth zone',
        style='Lake name',  markers=['d','o'],
        palette='magma',
        s=20,
        alpha=0.7,
    )

    #Set plot title and axis labels
    the_plot.set_axis_labels('Sample Date', nutrient)
    the_plot.set(title=f'{nutrient}')
    the_plot.set_xticklabels(rotation=45, ha='right')

    #Clean up the legend
    sns.move_legend(
        the_plot, 
        loc = 'center right',
        frameon = True,
    )

    #return the plot
    return the_plot

### Widgets
We'll create a set of widgets used to select data more interactively. `ipywidget` provides the following widgets to include: https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html

We will create the following. 
* [Dropdown widget](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#dropdown) to select the **nutrient** to plot
* A [selection range slider](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#selectionrangeslider) to select the **date range** of data to plot
* A [select mutiple widget](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html#selectmultiple) to select which **depth ranges** to include in the plot.
* Another select multiple widget to select which **lakes** to plot.

In [ ]:
#Create a dropdown widget from nutrient values
nutrient_list = df_nutrients['nutrient'].unique()
nutrient_dropdown = widgets.Dropdown(
    options=nutrient_list,
    value=nutrient_list[0],
    description='Nutrient:',
)

In [ ]:
#Create a date slider from the date range
all_dates = df_nutrients['sampledate'].sort_values().unique()
date_options = [(date.strftime("%Y/%m/%d"), date) for date in all_dates]

# Create a date range slider with start date labelled on the left and end date on the right
date_slider = widgets.SelectionRangeSlider(
    options = date_options,
    index = (0,len(date_options) - 1),
    description = 'Date range:',
    layout = widgets.Layout(width='75%')
)

In [ ]:
#Create a widget to select one or more depth zones via check box
depth_zones = df_nutrients['Depth zone'].unique()
depth_zone_select = widgets.SelectMultiple(
    options=depth_zones,
    value=(depth_zones[0],),
    description='Depth zones:',
)

In [ ]:
#Create a check box widget for each lake name
lake_names = df_nutrients['Lake name'].unique()
lake_name_select = widgets.SelectMultiple(
    options=lake_names,
    value=(lake_names[0],),
    description='Lake names:',
)

## Add reactivity to the widgets

In [ ]:
# Create an output area for the plot
plot_output = widgets.Output()

# Function to update the plot
def update_data_or_plot(change=None):
    #Direct output to the plot_output widget
    with plot_output:
        # Clear the previous plot
        clear_output(wait=True)

        # Generate the new plot
        g = plot_nutrients(
            start_date=date_slider.value[0],
            end_date=date_slider.value[1],
            nutrient=nutrient_dropdown.value,
            depth_zones=depth_zone_select.value,
            lake_names=lake_name_select.value
        )

        # Explicitly display the plot
        display(g.figure)
        plt.close(g.figure)

# Attach observers
date_slider.observe(update_data_or_plot, names='value')
nutrient_dropdown.observe(update_data_or_plot, names='value')
depth_zone_select.observe(update_data_or_plot, names='value')
lake_name_select.observe(update_data_or_plot, names='value')

# Display the widgets and plot together
display(
    widgets.VBox([
        date_slider,
        nutrient_dropdown,
        depth_zone_select,
        lake_name_select,
        plot_output
    ])
)

# Generate the initial plot
update_data_or_plot()